In [1]:
import pathlib
from decouple import config
from replicate.client import Client

In [2]:
NBS_DIR = pathlib.Path().resolve()
REPO_DIR = NBS_DIR.parent
DATA_DIR = REPO_DIR / "data"
GENERATED_DIR = DATA_DIR / "generated"
GENERATED_DIR.mkdir(exist_ok=True, parents=True)

REPLICATE_API_TOKEN = config("REPLICATE_API_TOKEN")
REPLICATE_MODEL = config("REPLICATE_MODEL", default="madhusanka-slc/super-slc-v1")
REPLICATE_MODEL_VERSION = config("REPLICATE_MODEL_VERSION", default="db2ecb62be5357f4673bdb959aea07650843f500837645d79e13965be242660b")

In [3]:
replicate_client = Client(api_token=REPLICATE_API_TOKEN)



In [5]:
model = f"{REPLICATE_MODEL}:{REPLICATE_MODEL_VERSION}"
prompt = "a photo of TOK adult man dressed up for a village man photo shoot"
num_outputs = 2
output_format = "jpg"

input_args = {
    "prompt": prompt,
    "num_outputs": 2,
    "output_format": "jpg",
}

In [6]:
rep_model = replicate_client.models.get(REPLICATE_MODEL)
rep_version = rep_model.versions.get(REPLICATE_MODEL_VERSION)

pred = replicate_client.predictions.create(
    version=rep_version,
    input=input_args
)



In [7]:
pred.id

'2bq9snqpnxrmt0cwnskstnwcs4'

In [8]:
pred.status

'starting'

In [10]:
# upstash -> qstash



In [11]:
pred_id = "2bq9snqpnxrmt0cwnskstnwcs4"
pred_lookup = replicate_client.predictions.get(pred_id)



In [12]:
pred_lookup.status

'succeeded'

In [13]:
pred_urls = pred_lookup.output
pred_urls

['https://replicate.delivery/xezq/61SPBdvsIt6RPRuptJveZUuZvf8L8kzFFh4VyKUak4FwzyMWA/out-0.jpg',
 'https://replicate.delivery/xezq/cpkRNgprfI3eN0Oj17SA3h1bDzk5oafAYx68v592bj2hnlZsA/out-1.jpg']

In [14]:
import httpx
import random

session_id = random.randint(1_000, 40_000)
with httpx.Client() as client:
    for i, url in enumerate(pred_urls):
        fname = f"{i}-{session_id}.jpg"
        outpath = GENERATED_DIR / fname
        res = client.get(url)
        res.raise_for_status()
        with open(outpath, 'wb') as f:
            f.write(res.content)